# ATiG 2026: LD score regression
29 October 2026 · About 75 minutes · Work in pairs

We estimate SNP heritability and genetic correlations from published GWAS results with
LD score regression (LDSC). Run the cells in order, discuss the questions and change the
settings marked `# TRY`. [Open in Colab](https://colab.research.google.com/github/bvilhjal/ATIG_2026/blob/main/teaching_days/2026-10-29/LDSC/LDSC_practical.ipynb) and choose
File → Save a copy in Drive.

## Setup
Run the three cells. The first installs [PLDSC](https://github.com/bvilhjal/PLDSC) and
[allelign](https://github.com/bvilhjal/allelign); the last downloads the GWAS files (about 240 MB).

In [ ]:
%pip install -q "pldsc @ git+https://github.com/bvilhjal/PLDSC@v0.1.1"

In [ ]:
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import allelign
from pldsc import load_ld_reference, ldsc_chi2_keep, ldsc_h2_sumstats, ldsc_rg_sumstats

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
#@title Download the GWAS files (no editing needed)
SOURCES = {  # trait: (file name, URL, SHA-256 of the file used here)
    'BMI': ('bmi_giant_ukb_2018.txt.gz',
            'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/'
            'GCST006001-GCST007000/GCST006900/Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz',
            '0d6ed0ea97870916b830ccae349df94ca6f3cc68c025c79e784729af7f7136a4'),
    'TG': ('tg_glgc_2013.txt.gz',
           'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_TG.txt.gz',
           '37087a0015c3211f5696616ad276eb4b5fae7219e0b8a2f37a5ed86b626c5f0b'),
    'DBP': ('dbp_evangelou_2018.txt.gz',
            'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/'
            'GCST006001-GCST007000/GCST006630/Evangelou_30224653_DBP.txt.gz',
            'c55cfc8b50abee8e13ef88d70ab52678b1f45b45cf5974bdab91f8c8b361fd59'),
}
folder = Path('gwas_downloads')
folder.mkdir(exist_ok=True)
paths = {}
for trait, (name, url, sha256) in SOURCES.items():
    path = folder / name
    if not path.exists():
        urllib.request.urlretrieve(url, folder / 'partial')
        (folder / 'partial').rename(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha256:
        raise ValueError(f'{path} is not the file used here; delete it and rerun')
    paths[trait] = path
    print(f'{trait}: {name}, {path.stat().st_size / 1e6:.0f} MB')

## Q1. Look at the files

BMI from [Yengo et al. 2018](https://pubmed.ncbi.nlm.nih.gov/30124842/) (about 700,000 people),
triglycerides (TG) from [Willer et al. 2013](https://pubmed.ncbi.nlm.nih.gov/24097068/) (about 90,000)
and diastolic blood pressure (DBP) from [Evangelou et al. 2018](https://pubmed.ncbi.nlm.nih.gov/30224653/)
(about 750,000).

Find the effect allele, other allele, effect, standard error, sample size and position in
each file. How do the files differ?

In [ ]:
for number, (trait, path) in enumerate(paths.items(), start=1):
    print(f'Table {number}. The first rows of the {trait} file, as text.')
    display(pd.read_csv(path, sep=r'\s+', nrows=3, dtype=str))

## Q2. Read the files

allelign recognises common column names (Table 4). The DBP file has no rsIDs, so we take
the positions from `MarkerName`. For DBP, allelign uses `N_effective`, the sample size
times imputation quality.

Which columns were not found? Why does N matter for heritability? Which trait has the
strongest associations in Figure 1, and why do the curves leave the diagonal?

In [ ]:
mappings = {trait: allelign.inspect_columns(path)['mapping'] for trait, path in paths.items()}
print('Table 4. File columns used by allelign; blank if not found.')
display(pd.DataFrame(mappings).fillna(''))

In [ ]:
gwas = {trait: allelign.read_sumstats(path) for trait, path in paths.items()}  # About 30 s.

# Chromosome and position from DBP's MarkerName, e.g. 10:100000625:SNP.
dbp = gwas['DBP']
marker = pd.Series(dbp.id).str.split(':', expand=True)
gwas['DBP'] = allelign.Sumstats(
    id=dbp.id, chrom=marker[0].to_numpy(), pos=marker[1].astype(int).to_numpy(),
    ea=dbp.ea, oa=dbp.oa, beta=dbp.beta, se=dbp.se, n_eff=dbp.n_eff, eaf=dbp.eaf, info=dbp.info)

for trait, sumstats in gwas.items():
    print(f'{trait}: {len(sumstats):,} variants, median N {np.median(sumstats.n_eff):,.0f}')

In [ ]:
plt.figure(figsize=(4.5, 4.5))
for trait, sumstats in gwas.items():
    z = np.sort(np.abs(sumstats.beta / sumstats.se))[::-1]  # Largest first.
    observed = -(stats.norm.logsf(z) + np.log(2)) / np.log(10)  # -log10 P, two-sided.
    rank = np.unique(np.geomspace(1, z.size, 3000).astype(int))  # Dense in the tail.
    expected = -np.log10((rank - 0.5) / z.size)
    plt.scatter(expected, np.minimum(observed[rank - 1], 40), s=6, label=trait)
plt.axline((0, 0), slope=1, color='grey', linestyle='--')
plt.xlabel('Expected −log10 P')
plt.ylabel('Observed −log10 P (capped at 40)')
plt.title('Figure 1. QQ plot of all variants')
plt.legend()
plt.tight_layout()
plt.show()

## Q3. Match to the LD reference

PLDSC comes with LD scores for about 1.2 million HapMap3 SNPs, calculated from 489
Europeans in the 1000 Genomes Project. It leaves out about 30,000 of them in regions with
long-range LD, such as the MHC. Each GWAS is aligned to the reference allele `a1` (Table 5).

Why is so little of the DBP file used? Table 6 follows rs1421085 in *FTO*. Why must the
traits refer to the same allele?

In [ ]:
ref = load_ld_reference('hm3-eur')
print(ref)

aligned = {}
counts = {}
for trait, sumstats in gwas.items():
    keep, qc = allelign.qc_sumstats(sumstats)
    aligned[trait] = allelign.harmonize(sumstats.subset(keep), ref.variants)
    log = aligned[trait].log
    counts[trait] = {'in file': len(sumstats), 'after QC': qc['n_kept'],
                     'matched': log['n_matched'], 'sign changed': log['n_flipped']}
print('Table 5. Variants kept at each step.')
display(pd.DataFrame(counts))

In [ ]:
snp = 'rs1421085'  # TRY: rs1260326 (GCKR) or rs13107325 (SLC39A8)
row = np.flatnonzero(ref.variants.id == snp)[0]
chrom, pos = ref.variants.chrom[row], ref.variants.pos[row]
print(f'{snp}: chromosome {chrom}, position {pos:,}; '
      f'reference a1 = {ref.variants.a1[row]}, a2 = {ref.variants.a2[row]}')

rows = {}
for trait, sumstats in gwas.items():
    # rsID for BMI and TG, position for DBP.
    i = np.flatnonzero((sumstats.id == snp) | ((sumstats.chrom == chrom) & (sumstats.pos == pos)))[0]
    j = np.flatnonzero(aligned[trait].var_index == row)[0]
    rows[trait] = {'effect allele': sumstats.ea[i], 'other allele': sumstats.oa[i],
                   'z in file': sumstats.beta[i] / sumstats.se[i],
                   'z for a1': aligned[trait].beta[j] / aligned[trait].se[j]}
snp_table = pd.DataFrame(rows).T.astype({'z in file': float, 'z for a1': float})
print(f'Table 6. {snp} in the three GWAS.')
display(snp_table)

## Q4. Estimate SNP heritability

LDSC models the χ² statistic of SNP $j$ as

$$E[\chi^2_j] = 1 + Na + \frac{N h^2}{M}\,\ell_j, \qquad (1)$$

with LD score $\ell_j$, sample size $N$, number of SNPs $M$ and confounding $a$.

Which trait has the highest heritability (Table 7, Figure 2)? Compare the intercepts with
λGC, the median χ² divided by 0.455. Why is TG's intercept below one? Set
`M = ref.m_snps` and rerun both cells: what changes?

In [ ]:
M = ref.m_snps_5_50  # SNPs with MAF 5–50%, as in the original LDSC. TRY: ref.m_snps

fits = {trait: ldsc_h2_sumstats(sumstats, ref.variants, ref.scores, m_snps=M).estimate
        for trait, sumstats in gwas.items()}  # About 30 s.

table = {}
for trait, fit in fits.items():
    z = aligned[trait].beta / aligned[trait].se
    table[trait] = {'h2': fit.h2, 'SE': fit.h2_se, 'intercept': fit.intercept,
                    'intercept SE': fit.intercept_se, 'mean chi2': fit.mean_chisq,
                    'lambda GC': allelign.genomic_inflation(z=z)}
print(f'Table 7. SNP heritability with M = {M:,}.')
display(pd.DataFrame(table).T)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, (trait, fit) in zip(axes, fits.items()):
    h = aligned[trait]
    chi2 = (h.beta / h.se)**2
    used = ldsc_chi2_keep(chi2, h.n_eff)[0]  # The fit drops SNPs with very large chi2.
    snps = pd.DataFrame({'ld_score': ref.scores[h.var_index][used], 'chi2': chi2[used]})
    snps = snps[snps['ld_score'] < snps['ld_score'].quantile(0.99)]  # Readable x-axis.
    snps['bin'] = pd.qcut(snps['ld_score'], 50, labels=False)  # 50 bins of equal size.
    means = snps.groupby('bin').mean()
    ax.scatter(means['ld_score'], means['chi2'], s=10)

    # Equation (1) with the fitted intercept and h2, at the median sample size.
    grid = np.linspace(0, means['ld_score'].max(), 100)
    ax.plot(grid, fit.intercept + np.median(h.n_eff) * fit.h2 / M * grid, color='darkorange')
    ax.set_title(trait)
    ax.set_xlabel('LD score')
axes[0].set_ylabel('Mean χ²')
fig.suptitle('Figure 2. Mean χ² in LD-score bins, with the LDSC fit (orange)')
plt.tight_layout()
plt.show()

## Q5. BMI and TG

Bivariate LDSC regresses the product of the two z scores on LD score:

$$E[z_{1j} z_{2j}] = \frac{\sqrt{N_1 N_2}\,\rho_g}{M}\,\ell_j + \frac{N_s\,\rho}{\sqrt{N_1 N_2}}, \qquad (2)$$

where $\rho_g$ is the genetic covariance, $N_s$ the number of people in both GWAS and $\rho$
their phenotypic correlation. The genetic correlation is $r_g = \rho_g / \sqrt{h^2_1 h^2_2}$.

Is $r_g$ clearly above zero? What does the intercept tell you? Why is the plain correlation
of z scores so much smaller? Where does the line in Figure 3 meet the y-axis?

In [ ]:
pair = ('BMI', 'TG')
fit = ldsc_rg_sumstats(gwas[pair[0]], gwas[pair[1]], ref.variants, ref.scores,
                       m_snps=M, ncores=2).estimate
print(f'{pair[0]}–{pair[1]} genetic correlation: {fit.rg:.3f} (SE {fit.rg_se:.3f})')
print(f'Genetic covariance: {fit.gcov:.4f}')
print(f'Cross-trait intercept: {fit.gcov_intercept:.3f} (SE {fit.gcov_intercept_se:.3f})')

# For comparison: the plain correlation of z scores, ignoring LD.
shared, i, j = np.intersect1d(aligned[pair[0]].var_index, aligned[pair[1]].var_index,
                              return_indices=True)
z1 = aligned[pair[0]].beta[i] / aligned[pair[0]].se[i]
z2 = aligned[pair[1]].beta[j] / aligned[pair[1]].se[j]
print(f'Correlation of z scores over {len(shared):,} SNPs: {np.corrcoef(z1, z2)[0, 1]:.3f}')

In [ ]:
# Mean z1 * z2 in LD-score bins, for the SNPs used in the fit.
n1, n2 = aligned[pair[0]].n_eff[i], aligned[pair[1]].n_eff[j]
used = ldsc_chi2_keep(z1**2, n1)[0] & ldsc_chi2_keep(z2**2, n2)[0]
snps = pd.DataFrame({'ld_score': ref.scores[shared][used], 'product': (z1 * z2)[used]})
snps = snps[snps['ld_score'] < snps['ld_score'].quantile(0.99)]  # Readable x-axis.
snps['bin'] = pd.qcut(snps['ld_score'], 50, labels=False)
means = snps.groupby('bin').mean()

# Equation (2) with the fitted genetic covariance and intercept, at the median sample sizes.
grid = np.linspace(0, means['ld_score'].max(), 100)
slope = np.sqrt(np.median(n1) * np.median(n2)) * fit.gcov / M
plt.figure(figsize=(5, 3.5))
plt.scatter(means['ld_score'], means['product'], s=10)
plt.plot(grid, fit.gcov_intercept + slope * grid, color='darkorange')
plt.axhline(0, color='grey', linestyle=':')
plt.xlabel('LD score')
plt.ylabel(f'Mean z({pair[0]}) × z({pair[1]})')
plt.title(f'Figure 3. {pair[0]}–{pair[1]} z-score products, with the fit (orange)')
plt.tight_layout()
plt.show()

## Q6. Add blood pressure

UK Biobank is part of both the BMI and the DBP GWAS, and DBP was adjusted for BMI
(Table 8, Figure 4).

Did you expect the BMI–DBP result? How can adjusting for BMI change it? Why is its
cross-trait intercept close to zero?

In [ ]:
pairs = [('BMI', 'TG'), ('BMI', 'DBP'), ('TG', 'DBP')]
table = {}
for trait1, trait2 in pairs:  # About a minute.
    estimate = ldsc_rg_sumstats(gwas[trait1], gwas[trait2], ref.variants, ref.scores,
                                m_snps=M, ncores=2).estimate
    table[f'{trait1}–{trait2}'] = {'rg': estimate.rg, 'SE': estimate.rg_se,
                                   'cross-trait intercept': estimate.gcov_intercept,
                                   'intercept SE': estimate.gcov_intercept_se}
print('Table 8. Genetic correlations.')
display(pd.DataFrame(table).T)

results = pd.DataFrame(table).T
plt.figure(figsize=(5, 2.5))
plt.errorbar(results['rg'], range(len(results)), xerr=1.96 * results['SE'], fmt='o')
plt.yticks(range(len(results)), results.index)
plt.gca().invert_yaxis()  # Same order as Table 8.
plt.axvline(0, color='grey', linestyle='--')
plt.xlabel('Genetic correlation, with 95% interval')
plt.title('Figure 4. Genetic correlations')
plt.tight_layout()
plt.show()

## Q7. Conclude

Write three sentences: the BMI–TG $r_g$ with its standard error, why it does not show that
BMI raises TG, and what the BMI–DBP result teaches you about published GWAS.

## Going further (optional)

- Run the cell below to add HDL, LDL, total cholesterol or height. Other European GWAS with
  rsIDs or GRCh37 positions work too, for example from the
  [GWAS Catalog](https://www.ebi.ac.uk/gwas/downloads/summary-statistics).
- Ask an AI model to write LDSC from scratch in NumPy from the arrays below, and compare
  with Table 7. Does it treat M, the weights and very large χ² the way PLDSC does?

```python
h = aligned['BMI']
z, n, ell = h.beta / h.se, h.n_eff, ref.scores[h.var_index]
```

In [ ]:
MORE = {  # Same formats as TG and BMI.
    'HDL': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_HDL.txt.gz',
    'LDL': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_LDL.txt.gz',
    'TC': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_TC.txt.gz',
    'Height': 'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/'
              'GCST006001-GCST007000/GCST006901/Meta-analysis_Wood_et_al+UKBiobank_2018.txt.gz',
}
trait = 'HDL'  # TRY: 'LDL', 'TC' or 'Height'
path = folder / MORE[trait].split('/')[-1]
if not path.exists():
    urllib.request.urlretrieve(MORE[trait], folder / 'partial')
    (folder / 'partial').rename(path)
gwas[trait] = allelign.read_sumstats(path)
print(trait, ldsc_h2_sumstats(gwas[trait], ref.variants, ref.scores, m_snps=M).estimate)
for other in ('BMI', 'TG', 'DBP'):
    fit = ldsc_rg_sumstats(gwas[other], gwas[trait], ref.variants, ref.scores,
                           m_snps=M, ncores=2).estimate
    print(f'{other}–{trait}: rg {fit.rg:.3f} (SE {fit.rg_se:.3f})')

## Data and methods

BMI ([GCST006900](https://www.ebi.ac.uk/gwas/studies/GCST006900)), DBP
([GCST006630](https://www.ebi.ac.uk/gwas/studies/GCST006630)) and TG
([GLGC](https://csg.sph.umich.edu/willer/public/lipids2013/)) are European-ancestry
meta-analyses on GRCh37; see [`data/sources.json`](data/sources.json). The TG and DBP results
were corrected for inflation before release. The LD scores are the 1000 Genomes Phase 3
European scores released with LDSC ([Zenodo 7768714](https://doi.org/10.5281/zenodo.7768714)).
PLDSC fits equations (1) and (2) in one step and drops SNPs with χ² above max(80, N/1000).
The original LDSC software uses a two-step estimator, so its estimates differ a little,
most for TG. Background: [Bulik-Sullivan et al. 2015a](https://pubmed.ncbi.nlm.nih.gov/25642630/)
and [2015b](https://pubmed.ncbi.nlm.nih.gov/26414676/).